# Shortcut Learning in Skin Lesion Classifiers — Setup

**Steps 1–4:** artifact correlation analysis, data loading, baseline classifier, masking delta.

**Runtime** → T4 GPU

## 0. Setup

In [ ]:
pip install -q grad-cam

import os, random, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('torch:', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything persists here so a disconnect doesn't lose your work
PROJECT = '/content/drive/MyDrive/shortcut-learning'
os.makedirs(PROJECT, exist_ok=True)
os.makedirs(f'{PROJECT}/checkpoints', exist_ok=True)
print(PROJECT)

## 1. Artifact correlation analysis

In [ ]:
!git clone -q --depth 1 https://github.com/alceubissoto/debiasing-skin.git /content/debiasing-skin

# NOTE: semicolon separator, and an unnamed index column
ann = pd.read_csv('/content/debiasing-skin/artefacts-annotation/isic_bias.csv',
                  sep=';', index_col=0)

ARTIFACTS = ['dark_corner','hair','gel_border','gel_bubble','ruler','ink','patches']

# strip .png so IDs join against the challenge .jpg files later
ann['image_id'] = ann['image'].str.replace('.png', '', regex=False)
ann['malignant'] = (ann['label_string'] == 'malignant').astype(int)

print(ann.shape)
ann.head()

In [ ]:
base = ann['malignant'].mean()

rows = []
for a in ARTIFACTS:
    present = ann[ann[a] == 1]
    absent  = ann[ann[a] == 0]
    rows.append({
        'artifact': a,
        'n_present': len(present),
        'prevalence': len(present) / len(ann),
        'malig_if_present': present['malignant'].mean(),
        'malig_if_absent': absent['malignant'].mean(),
        'lift': present['malignant'].mean() - base,
    })

corr_table = pd.DataFrame(rows).sort_values('lift', ascending=False)
print(f'Base malignancy rate: {base:.3f}  (n={len(ann)})\n')
display(corr_table.round(3))

corr_table.to_csv(f'{PROJECT}/artifact_correlations.csv', index=False)

In [ ]:
# a lift is only worth reporting if it survives this
from scipy.stats import chi2_contingency

for a in ARTIFACTS:
    table = pd.crosstab(ann[a], ann['malignant'])
    chi2, p, dof, _ = chi2_contingency(table)
    flag = '*' if p < 0.05 else ' '
    print(f'{a:12s} chi2={chi2:7.2f}  p={p:.2e} {flag}')

print('\n* = p < 0.05')

In [ ]:
# Figure 1
fig, ax = plt.subplots(figsize=(7, 4))
d = corr_table.sort_values('lift')
colors = ['#c0392b' if v > 0 else '#2874a6' for v in d['lift']]
ax.barh(d['artifact'], d['lift'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Malignancy rate lift vs. base rate')
ax.set_title(f'Artifact–label association, ISIC 2018 (base rate = {base:.1%})')
plt.tight_layout()
plt.savefig(f'{PROJECT}/fig1_artifact_lift.png', dpi=150)
plt.show()

## 2. ISIC 2018 data

In [ ]:
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)

# unzip to local disk
!unzip -q -o {PROJECT}/raw/ISIC2018_Task1-2_Training_Input.zip -d {DATA}
!unzip -q -o {PROJECT}/raw/ISIC2018_Task1_Training_GroundTruth.zip -d {DATA}

IMG_DIR  = f'{DATA}/ISIC2018_Task1-2_Training_Input'
MASK_DIR = f'{DATA}/ISIC2018_Task1_Training_GroundTruth'

print('images:', len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')]))
print('masks :', len([f for f in os.listdir(MASK_DIR) if f.endswith('.png')]))

In [ ]:
# Join annotations to files on disk. Verify nothing is missing BEFORE training.
def img_path(iid):  return f'{IMG_DIR}/{iid}.jpg'
def mask_path(iid): return f'{MASK_DIR}/{iid}_segmentation.png'

ann['has_img']  = ann['image_id'].apply(lambda i: os.path.exists(img_path(i)))
ann['has_mask'] = ann['image_id'].apply(lambda i: os.path.exists(mask_path(i)))

print('annotated rows :', len(ann))
print('with image     :', ann.has_img.sum())
print('with mask      :', ann.has_mask.sum())

df = ann[ann.has_img & ann.has_mask].reset_index(drop=True)
print('\nusable:', len(df))

# If 'usable' is much less than 2594, stop and check the mask filename pattern
print('\nsample mask filenames:', sorted(os.listdir(MASK_DIR))[:3])

In [ ]:
# Stratified split. Stratify on label and artifact count 
from sklearn.model_selection import train_test_split

df['n_artifacts'] = df[ARTIFACTS].sum(axis=1)
df['strat'] = df['malignant'].astype(str) + '_' + (df['n_artifacts'] > 0).astype(str)

train_df, test_df = train_test_split(
    df, test_size=0.25, stratify=df['strat'], random_state=SEED)

print('train:', len(train_df), ' malignant:', train_df.malignant.mean().round(3))
print('test :', len(test_df),  ' malignant:', test_df.malignant.mean().round(3))

train_df.to_csv(f'{PROJECT}/train_split.csv', index=False)
test_df.to_csv(f'{PROJECT}/test_split.csv', index=False)

## 3. Baseline classifier

In [ ]:
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# No augmentation at eval — your ablation depends on deterministic predictions
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class LesionDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(img_path(row.image_id)).convert('RGB')
        return self.transform(img), torch.tensor(row.malignant, dtype=torch.float32)

train_loader = DataLoader(LesionDataset(train_df, train_tf), batch_size=32,
                          shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(LesionDataset(test_df, eval_tf), batch_size=32,
                          shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_loader))
print('batch:', xb.shape, yb.shape)

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(DEVICE)

# 20% malignant -> weight the positive class or it predicts all-benign
pos_weight = torch.tensor([(1 - train_df.malignant.mean()) / train_df.malignant.mean()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print('pos_weight:', pos_weight.item().__round__(2))

In [ ]:
# Evaluate model on test set and return AUC and accuracy - Takes ~30 minutes on Colab T4 GPU
from sklearn.metrics import roc_auc_score

def evaluate(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.sigmoid(model(x.to(DEVICE))).squeeze(1).cpu()
            probs.append(p); labels.append(y)
    probs = torch.cat(probs).numpy(); labels = torch.cat(labels).numpy()
    return roc_auc_score(labels, probs), ((probs > 0.5) == labels).mean()

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x).squeeze(1), y)
        loss.backward(); optimizer.step()
        total += loss.item() * x.size(0)
    auc, acc = evaluate(model, test_loader)
    print(f'epoch {epoch+1}  loss={total/len(train_loader.dataset):.4f}  '
          f'test AUC={auc:.3f}  acc={acc:.3f}')

torch.save(model.state_dict(), f'{PROJECT}/checkpoints/resnet18_baseline.pth')
print('\nsaved checkpoint')

## 4. The masking delta — the core measurement

`delta = p(original) − p(masked)`

In [ ]:
def predict_prob(model, pil_img):
    model.eval()
    x = eval_tf(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return torch.sigmoid(model(x)).item()

def load_mask(image_id, size):
    m = Image.open(mask_path(image_id)).convert('L').resize(size, Image.NEAREST)
    return np.array(m) > 127

def apply_mask(pil_img, keep_bool, fill='blur'):
    arr = np.array(pil_img).astype(np.float32)
    if fill == 'blur':
        rep = np.array(pil_img.filter(ImageFilter.GaussianBlur(25))).astype(np.float32)
    elif fill == 'mean':
        rep = np.zeros_like(arr) + arr.reshape(-1, 3).mean(axis=0)
    else:
        rep = np.zeros_like(arr)
    out = np.where(keep_bool[..., None], arr, rep)
    return Image.fromarray(out.astype(np.uint8))

def deltas_for(model, image_id, fill='blur'):
    img = Image.open(img_path(image_id)).convert('RGB')
    lesion = load_mask(image_id, img.size)

    p_orig = predict_prob(model, img)
    p_no_lesion = predict_prob(model, apply_mask(img, ~lesion, fill))  # lesion removed
    p_no_bg     = predict_prob(model, apply_mask(img,  lesion, fill))  # background removed

    return {
        'image_id': image_id,
        'p_orig': p_orig,
        'p_lesion_masked': p_no_lesion,
        'p_background_masked': p_no_bg,
        'delta_lesion': p_orig - p_no_lesion,
        'delta_background': p_orig - p_no_bg,
    }

In [ ]:
# Smoke test on 5 pictures
for iid in test_df.image_id.head(5):
    r = deltas_for(model, iid)
    print(f"{r['image_id']}  p={r['p_orig']:.3f}  "
          f"Δlesion={r['delta_lesion']:+.3f}  Δbackground={r['delta_background']:+.3f}")

In [ ]:
iid = test_df.image_id.iloc[0]
img = Image.open(img_path(iid)).convert('RGB')
lesion = load_mask(iid, img.size)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (im, title) in zip(axes, [
        (img, 'original'),
        (apply_mask(img, ~lesion), 'lesion masked'),
        (apply_mask(img,  lesion), 'background masked')]):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()